# Task 2 - Word2Vec From Scratch

This notebook trains a small skip-gram Word2Vec model with negative sampling using NumPy only. It uses the cleaned IMDB data saved in `data/preprocessed/imdb`.

The training data is prepared before the model is created. The model receives numeric skip-gram pairs and word counts through `fit(...)`.

<!-- task2_settings_answer -->
## Notebook Settings

This notebook uses the cleaned IMDB train split from `data/preprocessed/imdb`. The main experiment uses `REVIEW_SAMPLE_SIZE = 2000`, `VOCAB_SIZE = 3000`, `MIN_COUNT = 2`, `WINDOW_SIZE = 2`, `EMBEDDING_DIM = 100`, `NEGATIVE_SAMPLES = 5`, `LEARNING_RATE = 0.01`, `N_EPOCHS = 10`, and `DIMENSION_EPOCHS = 5`. The fixed `SEED = 42` comes from `src.config` so the sample and model initialization stay reproducible. For the dimension comparison, the notebook tries `50`, `100`, and `200` dimensions on the same prepared training pairs.

## Setup

In [1]:
from pathlib import Path
import random
import sys

from IPython.display import Markdown, display
from datasets import load_from_disk
import numpy as np
import pandas as pd
import torch

/home/atanas-vitanov/Documents/praksa/ml-interns-mk-2026/interns/atanas/week_6/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

In [3]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
from src.config import METADATA_DIR, PLOTS_DIR, PREPROCESSED_DATA_DIR, SEED, TABLES_DIR
from src.plotting import save_line_plot
from src.result_saver import save_dataframe, save_json
from src.word2vec_numpy import (
    SkipGramWord2Vec,
    build_vocabulary,
    build_word_counts,
    encode_sentences,
    generate_skipgram_pairs,
    get_nearest_neighbors,
    run_analogy,
    tokenize_texts,
)

In [5]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [6]:
for output_dir in [METADATA_DIR, PLOTS_DIR, TABLES_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

## Load Preprocessed IMDB Data

In [7]:
IMDB_PATH = PREPROCESSED_DATA_DIR / "imdb"
imdb_dataset = load_from_disk(str(IMDB_PATH))

In [8]:
imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 24902
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 24676
    })
})

In [9]:
REVIEW_SAMPLE_SIZE = 2000

imdb_sample = (
    imdb_dataset["train"]
    .shuffle(seed=SEED)
    .select(range(REVIEW_SAMPLE_SIZE))
)

In [10]:
print("sample size:", len(imdb_sample))
print("columns:", imdb_sample.column_names)
print(imdb_sample[0]["text"])

sample size: 2000
columns: ['text', 'label']
So they hyped the violence and it's been branded as sick. Well, the violence is the best bit I'm afraid, but unfortunately the characters are not developed enough to allow us to understand why they go on their (entirely predictable) rampage. This film has a truly dreadful script. We never get a chance to get to know Robert and his actions at the end are just plain pathetic. The acting isn't much better, either, the worst of them being the TV chef and the school teacher. The direction is clumsy, the pace enough to send you to sleep. And what on earth is the school film project all about? A comment on the film itself perhaps? The use of newsreel during the climactic murder is laughable. These guys obviously think they're intellectuals but are hopelessly out of their depth. How on earth they got the great Yorgos Arvanitis to light it I'll never know. And how they got the money to make it in the first place is an even greater mystery. Absolutely

## Prepare Tokens

This step turns each review into lowercase word tokens. The reviews are **not truncated**, so the training pairs come from the full selected text.

In [11]:
texts = list(imdb_sample["text"])

In [12]:
tokenized_reviews = tokenize_texts(texts=texts)

In [13]:
token_lengths = pd.Series([len(tokens) for tokens in tokenized_reviews])
token_lengths.describe()

count    2000.00000
mean      232.86100
std       172.84714
min        24.00000
25%       127.00000
50%       173.00000
75%       275.25000
max      1285.00000
dtype: float64

<!-- task2_token_observation -->
**Observation:** The selected reviews are not all the same length. The average review has about **233 tokens**, while the longest selected review has **1,285 tokens**. This matters because longer reviews create many more training pairs.

In [14]:
tokenized_reviews[0]

['so',
 'they',
 'hyped',
 'the',
 'violence',
 'and',
 "it's",
 'been',
 'branded',
 'as',
 'sick',
 'well',
 'the',
 'violence',
 'is',
 'the',
 'best',
 'bit',
 "i'm",
 'afraid',
 'but',
 'unfortunately',
 'the',
 'characters',
 'are',
 'not',
 'developed',
 'enough',
 'to',
 'allow',
 'us',
 'to',
 'understand',
 'why',
 'they',
 'go',
 'on',
 'their',
 'entirely',
 'predictable',
 'rampage',
 'this',
 'film',
 'has',
 'a',
 'truly',
 'dreadful',
 'script',
 'we',
 'never',
 'get',
 'a',
 'chance',
 'to',
 'get',
 'to',
 'know',
 'robert',
 'and',
 'his',
 'actions',
 'at',
 'the',
 'end',
 'are',
 'just',
 'plain',
 'pathetic',
 'the',
 'acting',
 "isn't",
 'much',
 'better',
 'either',
 'the',
 'worst',
 'of',
 'them',
 'being',
 'the',
 'tv',
 'chef',
 'and',
 'the',
 'school',
 'teacher',
 'the',
 'direction',
 'is',
 'clumsy',
 'the',
 'pace',
 'enough',
 'to',
 'send',
 'you',
 'to',
 'sleep',
 'and',
 'what',
 'on',
 'earth',
 'is',
 'the',
 'school',
 'film',
 'project',
 '

## Build Vocabulary

The vocabulary keeps the most frequent **3,000** words. `MIN_COUNT = 2` removes words that appear only once, and words outside the vocabulary become `<UNK>`.

In [15]:
VOCAB_SIZE = 3000
MIN_COUNT = 2

vocabulary, id_to_word, raw_token_counts = build_vocabulary(
    tokenized_texts=tokenized_reviews,
    vocab_size=VOCAB_SIZE,
    min_count=MIN_COUNT,
)

In [16]:
id_to_word

{0: '<UNK>',
 1: 'the',
 2: 'and',
 3: 'a',
 4: 'of',
 5: 'to',
 6: 'is',
 7: 'in',
 8: 'i',
 9: 'it',
 10: 'this',
 11: 'that',
 12: 'was',
 13: 'as',
 14: 'with',
 15: 'for',
 16: 'movie',
 17: 'but',
 18: 'film',
 19: 'on',
 20: 'his',
 21: 'you',
 22: 'not',
 23: 'are',
 24: 'he',
 25: 'have',
 26: 'one',
 27: 'be',
 28: 'all',
 29: 'at',
 30: 'by',
 31: 'so',
 32: 'they',
 33: 'an',
 34: 'from',
 35: 'like',
 36: 'who',
 37: 'her',
 38: 'about',
 39: 'or',
 40: 'if',
 41: 'out',
 42: 'just',
 43: "it's",
 44: 'has',
 45: 'some',
 46: 'there',
 47: 'what',
 48: 'more',
 49: 'good',
 50: 'when',
 51: 'she',
 52: 'very',
 53: 'up',
 54: 'my',
 55: 'time',
 56: 'even',
 57: 'would',
 58: 'really',
 59: 'no',
 60: 'only',
 61: 'see',
 62: 'story',
 63: 'were',
 64: 'which',
 65: 'me',
 66: 'had',
 67: 'their',
 68: 'can',
 69: 'well',
 70: 'we',
 71: 'much',
 72: 'than',
 73: 'get',
 74: 'him',
 75: 'also',
 76: 'people',
 77: 'bad',
 78: 'great',
 79: 'first',
 80: 'do',
 81: 'how',
 

In [17]:
print("vocabulary size:", len(vocabulary))
print("first 20 vocabulary items:")
list(vocabulary.items())[:20]

vocabulary size: 3000
first 20 vocabulary items:


[('<UNK>', 0),
 ('the', 1),
 ('and', 2),
 ('a', 3),
 ('of', 4),
 ('to', 5),
 ('is', 6),
 ('in', 7),
 ('i', 8),
 ('it', 9),
 ('this', 10),
 ('that', 11),
 ('was', 12),
 ('as', 13),
 ('with', 14),
 ('for', 15),
 ('movie', 16),
 ('but', 17),
 ('film', 18),
 ('on', 19)]

In [18]:
top_words = pd.DataFrame(
    list(raw_token_counts.items()),
    columns=["word", "count"],
).sort_values("count", ascending=False)

top_words.head(10)

,word,count
3,the,26703
5,and,13023
38,a,12919
61,of,11563
24,to,10996
12,is,8469
107,in,7455
165,i,6293
103,it,6285
35,this,5951


<!-- task2_vocab_observation -->
**Observation:** The most common words are mostly general English words such as `the`, `and`, `a`, `of`, and `to`. This is normal for text data. The vocabulary still includes movie-specific words like `movie` and `film`, but many rare words are grouped under `<UNK>`.

## Generate Training Pairs

The encoded sentences and skip-gram pairs are prepared before the model is created. This keeps data preparation separate from model training, so `fit(...)` receives ready-made numeric pairs.

In [19]:
encoded_sentences = encode_sentences(
    tokenized_texts=tokenized_reviews,
    vocabulary=vocabulary,
)

In [20]:
word_counts = build_word_counts(
    encoded_sentences=encoded_sentences,
    vocab_size=len(vocabulary),
)

In [21]:
print("encoded sentence count:", len(encoded_sentences))
print("first encoded sentence length:", len(encoded_sentences[0]))
print("first 20 ids:", encoded_sentences[0][:20])
print("unknown token count:", int(word_counts[vocabulary["<UNK>"]]))

encoded sentence count: 2000
first encoded sentence length: 176
first 20 ids: [31, 32, 0, 1, 641, 2, 43, 82, 0, 13, 1155, 69, 1, 641, 6, 1, 114, 230, 158, 1711]
unknown token count: 61894


<!-- task2_encoding_observation -->
**Observation:** The encoded data still has **2,000** reviews. The `<UNK>` count is **61,894**, which shows how many token positions were rare or outside the top 3,000-word vocabulary. This is a tradeoff: the model stays smaller, but some rare word detail is lost.

In [22]:
WINDOW_SIZE = 2

training_pairs = generate_skipgram_pairs(
    encoded_sentences=encoded_sentences,
    window_size=WINDOW_SIZE,
)

In [23]:
training_pairs

array([[  31,   32],
       [  31,    0],
       [  32,   31],
       ...,
       [1811, 1699],
       [1699,    5],
       [1699, 1811]], shape=(1850888, 2))

In [24]:
print("training pair shape:", training_pairs.shape)

pair_preview = pd.DataFrame(training_pairs[:10], columns=["center_id", "context_id"])
pair_preview["center_word"] = pair_preview["center_id"].map(id_to_word)
pair_preview["context_word"] = pair_preview["context_id"].map(id_to_word)
pair_preview

training pair shape: (1850888, 2)


,center_id,context_id,center_word,context_word
0,31,32,so,they
1,31,0,so,<UNK>
2,32,31,they,so
3,32,0,they,<UNK>
4,32,1,they,the
5,0,31,<UNK>,so
6,0,32,<UNK>,they
7,0,1,<UNK>,the
8,0,641,<UNK>,violence
9,1,32,the,they


<!-- task2_pairs_answer -->
**Question: How many training pairs were generated?**

The notebook generated **1,850,888** center-context pairs from the 2,000 selected IMDB reviews. Each pair is one learning example: the model sees a center word and one nearby context word. With `window_size = 2`, most words can be paired with up to two words on the left and two words on the right.

In [25]:
task2_config = {
    "task": "Task 2 - Word2Vec from Scratch",
    "dataset": "Preprocessed IMDB train split",
    "sample_size": REVIEW_SAMPLE_SIZE,
    "vocab_size": len(vocabulary),
    "min_count": MIN_COUNT,
    "window_size": WINDOW_SIZE,
    "training_pair_count": int(len(training_pairs)),
    "seed": SEED
}

save_json(
    data=task2_config,
    output_path=METADATA_DIR / "task2_word2vec_config.json",
)

In [26]:
task2_config

{'task': 'Task 2 - Word2Vec from Scratch',
 'dataset': 'Preprocessed IMDB train split',
 'sample_size': 2000,
 'vocab_size': 3000,
 'min_count': 2,
 'window_size': 2,
 'training_pair_count': 1850888,
 'seed': 42}

## Train Skip-Gram Word2Vec

The model uses two embedding matrices and negative sampling. Training updates one center-context pair at a time; negative words are sampled from the unigram distribution raised to `0.75`.

In [27]:
EMBEDDING_DIM = 100
NEGATIVE_SAMPLES = 5
LEARNING_RATE = 0.01
N_EPOCHS = 10

word2vec_model = SkipGramWord2Vec(
    vocab_size=len(vocabulary),
    embedding_dim=EMBEDDING_DIM,
    negative_samples=NEGATIVE_SAMPLES,
    learning_rate=LEARNING_RATE,
    n_epochs=N_EPOCHS,
    seed=SEED,
)

In [28]:
print("embedding matrix shape:", word2vec_model.input_embeddings.shape)
print("negative samples:", word2vec_model.negative_samples)
print("epochs:", word2vec_model.n_epochs)

embedding matrix shape: (3000, 100)
negative samples: 5
epochs: 10


In [29]:
word2vec_model.fit(
    training_pairs=training_pairs,
    word_counts=word_counts,
)

In [30]:
loss_frame = pd.DataFrame({
    "epoch": list(range(1, len(word2vec_model.losses) + 1)),
    "loss": word2vec_model.losses,
})

loss_frame

,epoch,loss
0,1,2.728653
1,2,2.438834
2,3,2.401083
3,4,2.378451
4,5,2.360909
5,6,2.345903
6,7,2.334119
7,8,2.322990
8,9,2.313315
9,10,2.304137


<!-- task2_loss_answer -->
**Observation: Did training improve the model?**

Yes. The average loss decreased from **2.7287** in epoch 1 to **2.3041** in epoch 10. This means the model became better at recognizing real nearby words compared with sampled negative words. The loss is still not a direct human-quality score, but the downward trend shows that learning happened.

In [31]:
save_line_plot(
    x_values=loss_frame["epoch"].tolist(),
    y_values=loss_frame["loss"].tolist(),
    title="Task 2 Word2Vec Training Loss",
    x_label="Epoch",
    y_label="Average negative-sampling loss",
    output_path=PLOTS_DIR / "task2_training_loss.png",
)

In [32]:
print("saved plot:", PLOTS_DIR / "task2_training_loss.png")
print("final loss:", round(word2vec_model.losses[-1], 4))

saved plot: /home/atanas-vitanov/Documents/praksa/ml-interns-mk-2026/interns/atanas/week_6/results/plots/task2_training_loss.png
final loss: 2.3041


In [33]:
embeddings = word2vec_model.get_embeddings()

In [34]:
embeddings.shape

(3000, 100)

<!-- task2_embeddings_answer -->
**Question: What embeddings did we extract after training?**

After training, the notebook extracts `W_in`, the input embedding matrix, with shape **(3000, 100)**. This means every vocabulary item has one learned vector with 100 numbers. These vectors are then used for nearest-neighbor search and analogy checks.

## Nearest Neighbors

This check asks whether words used in similar review contexts ended up close together in the learned vector space.

In [35]:
probe_words = ["good", "bad", "movie", "actor", "great"]

In [36]:
nearest_neighbor_rows = []

for probe_word in probe_words:
    neighbors = get_nearest_neighbors(
        word=probe_word,
        embeddings=embeddings,
        vocabulary=vocabulary,
        id_to_word=id_to_word,
        top_n=5,
    )
    for rank, neighbor in enumerate(neighbors, start=1):
        nearest_neighbor_rows.append({
            "probe_word": probe_word,
            "rank": rank,
            "neighbor": neighbor["word"],
            "similarity": neighbor["similarity"],
        })

nearest_neighbors_df = pd.DataFrame(nearest_neighbor_rows)

In [37]:
nearest_neighbors_df

,probe_word,rank,neighbor,similarity
0,good,1,decent,0.688133
1,good,2,terrific,0.656970
2,good,3,crappy,0.649990
3,good,4,creepy,0.645356
4,good,5,fine,0.645194
5,bad,1,scary,0.704225
6,bad,2,terrible,0.700018
7,bad,3,lame,0.699848
8,bad,4,horrible,0.683080
9,bad,5,rare,0.676249


<!-- task2_neighbors_answer -->
**Question: Do the nearest neighbors make sense?**

Mostly yes, with some noise. `movie` is closest to `film`, `mess`, `picture`, `flick`, and `documentary`, which are all review-related. `actor` is closest to `actress`, `role`, and `person`, which also makes sense. `bad` is close to words like `terrible`, `lame`, and `horrible`. The noisy part is visible for `good`, where words like `crappy` and `creepy` appear; that happens because this small model learns from nearby review language, not perfect dictionary meaning.

In [38]:
save_dataframe(
    dataframe=nearest_neighbors_df,
    output_path=TABLES_DIR / "task2_nearest_neighbors.csv",
)

## Analogies

Analogies are harder than nearest-neighbor lookup. They test whether vector arithmetic learned a useful relationship, not just whether related words are nearby.

In [39]:
analogy_questions = [
    ("good", "bad", "great"),
    ("man", "woman", "actor"),
    ("movie", "film", "actor"),
]

In [40]:
analogy_rows = []

for positive_word, negative_word, target_word in analogy_questions:
    candidates = run_analogy(
        positive_word=positive_word,
        negative_word=negative_word,
        target_word=target_word,
        embeddings=embeddings,
        vocabulary=vocabulary,
        id_to_word=id_to_word,
        top_n=5,
    )
    question = f"{positive_word} - {negative_word} + {target_word}"
    for rank, candidate in enumerate(candidates, start=1):
        analogy_rows.append({
            "analogy": question,
            "rank": rank,
            "candidate": candidate["word"],
            "similarity": candidate["similarity"],
        })

analogies_df = pd.DataFrame(analogy_rows)

In [41]:
analogies_df

,analogy,rank,candidate,similarity
0,good - bad + great,1,fine,0.571967
1,good - bad + great,2,wonderful,0.564228
2,good - bad + great,3,excellent,0.542805
3,good - bad + great,4,decent,0.530563
4,good - bad + great,5,portraying,0.530490
5,man - woman + actor,1,actress,0.667747
6,man - woman + actor,2,writer,0.607998
7,man - woman + actor,3,person,0.579262
8,man - woman + actor,4,soldier,0.562635
9,man - woman + actor,5,walken,0.560914


<!-- task2_analogies_answer -->
**Question: Did the analogy tasks work?**

Some worked in a simple movie-review way. `man - woman + actor` returned `actress` first, and `movie - film + actor` also returned `actress` first. `good - bad + great` returned positive words like `fine`, `wonderful`, and `excellent`, but it did not produce a clean textbook analogy. This is expected because the model only saw 2,000 movie reviews, not a large general-purpose corpus.

In [42]:
save_dataframe(
    dataframe=analogies_df,
    output_path=TABLES_DIR / "task2_analogies.csv",
)

## Embedding Dimension Check

This section trains three comparison models with different vector sizes: 50, 100, and 200. Each comparison uses the same training pairs and **5 epochs**, so the table focuses on how the vector size changes the result.

In [43]:
comparison_pairs = training_pairs

In [44]:
print("comparison pair shape:", comparison_pairs.shape)

comparison pair shape: (1850888, 2)


In [45]:
DIMENSION_EPOCHS = 5

dimension_rows = []

for embedding_dim in [50, 100, 200]:
    comparison_model = SkipGramWord2Vec(
        vocab_size=len(vocabulary),
        embedding_dim=embedding_dim,
        negative_samples=NEGATIVE_SAMPLES,
        learning_rate=LEARNING_RATE,
        n_epochs=DIMENSION_EPOCHS,
        seed=SEED,
    )
    comparison_model.fit(
        training_pairs=comparison_pairs,
        word_counts=word_counts,
    )
    comparison_embeddings = comparison_model.get_embeddings()
    movie_neighbors = get_nearest_neighbors(
        word="movie",
        embeddings=comparison_embeddings,
        vocabulary=vocabulary,
        id_to_word=id_to_word,
        top_n=5,
    )
    dimension_rows.append({
        "embedding_dim": embedding_dim,
        "epochs": DIMENSION_EPOCHS,
        "final_loss": comparison_model.losses[-1],
        "movie_neighbors": ", ".join([item["word"] for item in movie_neighbors]),
    })

dimension_comparison_df = pd.DataFrame(dimension_rows)

In [46]:
dimension_comparison_df

,embedding_dim,epochs,final_loss,movie_neighbors
0,50,5,2.362346,"film, documentary, picture, mess, flick"
1,100,5,2.360909,"film, picture, documentary, mess, show"
2,200,5,2.362354,"film, flick, picture, documentary, mess"


<!-- task2_dimension_answer -->
**Question: How did changing `embedding_dim` affect neighbor quality?**

Increasing the vector size did not clearly improve the result in this run. The **100-dimensional** model had the lowest final loss (**2.3609**), but the difference from 50 and 200 dimensions was very small. All three models kept `film` as the first neighbor for `movie`, and the rest of the neighbors stayed movie-related (`documentary`, `picture`, `mess`, `flick`, or `show`). A concrete takeaway is that more dimensions did not automatically make better embeddings for this small IMDB corpus.

## Save Training Summary

The summary saves the main numbers and interpretation so the result can be reused by the frontend or checked later without rerunning training.

In [47]:
best_dimension_row = dimension_comparison_df.sort_values("final_loss").iloc[0]
dimension_observation = (
    f"On the full-pair comparison, {int(best_dimension_row['embedding_dim'])} dimensions "
    f"had the lowest final loss ({best_dimension_row['final_loss']:.4f}). "
    "The nearest-neighbor words still changed across dimensions, which shows that the 2,000-review corpus is still noisy."
)

training_summary = {
    "task": "Task 2 - Word2Vec from Scratch",
    "dataset": "Preprocessed IMDB train split",
    "sample_size": REVIEW_SAMPLE_SIZE,
    "vocab_size": len(vocabulary),
    "training_pair_count": int(len(training_pairs)),
    "embedding_dim": EMBEDDING_DIM,
    "window_size": WINDOW_SIZE,
    "negative_samples": NEGATIVE_SAMPLES,
    "learning_rate": LEARNING_RATE,
    "epochs": N_EPOCHS,
    "dimension_epochs": DIMENSION_EPOCHS,
    "first_epoch_loss": float(word2vec_model.losses[0]),
    "final_epoch_loss": float(word2vec_model.losses[-1]),
    "loss_change": float(word2vec_model.losses[0] - word2vec_model.losses[-1]),
    "probe_words": probe_words,
    "analogy_questions": [
        f"{positive} - {negative} + {target}"
        for positive, negative, target in analogy_questions
    ],
    "dimension_observation": dimension_observation,
    "main_result": "The model learned local IMDB word associations, but analogy quality is limited by the domain-specific corpus.",
    "interpretation": "Training loss and neighbor tables are useful checks, but the model is not expected to match pretrained embeddings because it sees only a 2,000-review movie-review subset.",
}

save_json(
    data=training_summary,
    output_path=METADATA_DIR / "task2_word2vec_training_summary.json",
)

In [48]:
training_summary

{'task': 'Task 2 - Word2Vec from Scratch',
 'dataset': 'Preprocessed IMDB train split',
 'sample_size': 2000,
 'vocab_size': 3000,
 'training_pair_count': 1850888,
 'embedding_dim': 100,
 'window_size': 2,
 'negative_samples': 5,
 'learning_rate': 0.01,
 'epochs': 10,
 'dimension_epochs': 5,
 'first_epoch_loss': 2.72865287222179,
 'final_epoch_loss': 2.3041372216119367,
 'loss_change': 0.4245156506098535,
 'probe_words': ['good', 'bad', 'movie', 'actor', 'great'],
 'analogy_questions': ['good - bad + great',
  'man - woman + actor',
  'movie - film + actor'],
 'dimension_observation': 'On the full-pair comparison, 100 dimensions had the lowest final loss (2.3609). The nearest-neighbor words still changed across dimensions, which shows that the 2,000-review corpus is still noisy.',
 'main_result': 'The model learned local IMDB word associations, but analogy quality is limited by the domain-specific corpus.',
 'interpretation': 'Training loss and neighbor tables are useful checks, but th

## Interpretation

In [50]:
first_neighbors = nearest_neighbors_df.groupby("probe_word").first().reset_index()
neighbor_sentence = "; ".join(
    f"{row.probe_word} -> {row.neighbor}"
    for row in first_neighbors.itertuples()
)

interpretation_text = f'''
**What was computed:** I trained a NumPy skip-gram Word2Vec model on {REVIEW_SAMPLE_SIZE} cleaned IMDB reviews. The preprocessing step produced {len(vocabulary):,} vocabulary items and {len(training_pairs):,} center-context pairs.

**Training result:** The average negative-sampling loss moved from {word2vec_model.losses[0]:.4f} to {word2vec_model.losses[-1]:.4f}. This suggests the model learned some signal from local context instead of staying at its random initialization.

**Nearest-neighbor result:** The first neighbor for each probe was: {neighbor_sentence}. The neighbors mostly make sense for movie-review language: `movie` is closest to `film`, `actor` is closest to `actress`, and `great` is closest to `wonderful`. Some neighbors are still noisy, especially for sentiment words like `good`, so this is a corpus-specific sanity check rather than proof of strong semantic quality.

**Analogy limitation:** The analogy table is noisy because the model only saw a 2,000-review movie-review subset instead of a large general corpus. Strong analogies usually need much larger and more balanced text corpora.

**Dimension check:** {dimension_observation} In this run, changing the vector size did not change the main `movie` to `film` relationship.
'''

display(Markdown(interpretation_text))


**What was computed:** I trained a NumPy skip-gram Word2Vec model on 2000 cleaned IMDB reviews. The preprocessing step produced 3,000 vocabulary items and 1,850,888 center-context pairs.

**Training result:** The average negative-sampling loss moved from 2.7287 to 2.3041. This suggests the model learned some signal from local context instead of staying at its random initialization.

**Nearest-neighbor result:** The first neighbor for each probe was: actor -> actress; bad -> scary; good -> decent; great -> wonderful; movie -> film. The neighbors mostly make sense for movie-review language: `movie` is closest to `film`, `actor` is closest to `actress`, and `great` is closest to `wonderful`. Some neighbors are still noisy, especially for sentiment words like `good`, so this is a corpus-specific sanity check rather than proof of strong semantic quality.

**Analogy limitation:** The analogy table is noisy because the model only saw a 2,000-review movie-review subset instead of a large general corpus. Strong analogies usually need much larger and more balanced text corpora.

**Dimension check:** On the full-pair comparison, 100 dimensions had the lowest final loss (2.3609). The nearest-neighbor words still changed across dimensions, which shows that the 2,000-review corpus is still noisy. In this run, changing the vector size did not change the main `movie` to `film` relationship.


<!-- task2_plain_conclusion -->
## Conclusion

This experiment shows how a computer can learn word meaning by looking at nearby words in movie reviews. The model learned useful patterns: `movie` moved close to `film`, and `actor` moved close to `actress`. It also learned some sentiment patterns, but not perfectly, because words in reviews can appear in messy or sarcastic contexts. The analogy examples show that this small model can catch a few simple relationships, but it is not broad enough to understand language like a large pretrained model. In plain terms, the model learned something real from the reviews, but it is still a practice model with clear limits.